In [ ]:
# Install required packages (run once)
import subprocess, sys

print('📦 Installing required packages...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'langchain>=0.3.25',
    'langchain-aws>=0.2.24',
    'langchain-community>=0.3.23',
    'langchain-core>=0.3.62',
    'boto3>=1.38.0',
    'pydantic>=2.11.4',
], check=True)
print('✅ All packages installed!')

In [ ]:
import os, json, math
from typing import Any, Optional
from datetime import datetime

# Load API key from .env file (never hardcode secrets in notebooks!)
try:
    from dotenv import load_dotenv
    load_dotenv()
    print('✅ python-dotenv loaded')
except ImportError:
    print('⚠️  python-dotenv not installed — run: pip install python-dotenv')

from langchain_core.tools import Tool, tool
print('✅ langchain_core.tools imported')

from langchain_core.prompts import PromptTemplate
print('✅ langchain_core.prompts imported')

from langchain_aws import ChatBedrockConverse
from langchain_core.messages import HumanMessage
print('✅ langchain_aws imported')

# LangChain 1.x uses create_agent (replaces create_react_agent + AgentExecutor)
from langchain.agents import create_agent
print('✅ create_agent imported (LangChain 1.x API)')

# ── AWS Configuration ──────────────────────────────────────
# Key is read from .env file (AWS_BEARER_TOKEN_BEDROCK=your_key_here)
BEDROCK_API_KEY = os.getenv('AWS_BEARER_TOKEN_BEDROCK', '')
AWS_REGION = 'ap-southeast-2'
MODEL_ID   = 'global.amazon.nova-2-lite-v1:0'

if not BEDROCK_API_KEY:
    BEDROCK_API_KEY = input('⚠️  AWS_BEARER_TOKEN_BEDROCK not found in .env\nEnter your Bearer Token: ').strip()

os.environ['AWS_BEARER_TOKEN_BEDROCK'] = BEDROCK_API_KEY
os.environ['AWS_DEFAULT_REGION']       = AWS_REGION

print(f'\n✅ AWS environment configured')
print(f'   Region : {AWS_REGION}')
print(f'   Model  : {MODEL_ID}')
print('\n' + '='*70)
print('🚀 READY TO BUILD AGENTS! (LangChain 1.x)')
print('='*70)


In [ ]:
import math
from langchain_core.tools import tool


# Tool 1: calculate expense
@tool
def calculate_expense(amount: float, category: str, description: str) -> str:
    
    """
    Creates a formatted expense entry.

    Args:
        amount: The amount spent.
        category: Expense category such as Food, Transport, or Shopping.
        description: Description of the expense.

    Returns:
        A formatted expense entry with today's date and category.
    """
#     amount = int(float(str(amount).replace(",", "")).replace(".", ""))
    amount = str(amount).replace("$", "").replace(",", "").replace(".", "").strip()
    amount = int(amount)
    if amount <= 0:
          return 'Error: Expense amount must be greater than 0'
          
        
    # Clean the text 
    
    category = category.replace('$', '').replace(',', '').replace('\u2018', '').replace('\u2019', '').replace('\u201c', '').replace('\u201d', '').replace("'", '').replace('"', '').strip().title()
    
    description = description.replace('$', '').replace(',', '').replace('\u2018', '').replace('\u2019', '').replace('\u201c', '').replace('\u201d', '').replace("'", '').replace('"', '').strip()
    
    today = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    
 
    expense_entry = (
    
    f"Date: {today}\n"
    f"Expense amount: {amount:.0f}\n"
    f"category: {category}\n"
    f"description: {description}\n"
    
    )

    return expense_entry

EXCHANGE_RATES_TO_USD = {
    "USD": 1.0,
    "PKR": 1 / 280.0,   # 280 PKR = 1 USD
    "INR": 1 / 90.0,    # 90 INR = 1 USD
    "EUR": 1/ 1.10,        # 1 EUR = 1.10 USD
    "GBP": 1/ 1.30,        # 1 GBP = 1.30 USD
    "AED": 1 / 3.67,    # 3.67 AED = 1 USD
    "SAR": 1 / 3.75,    # 3.75 SAR = 1 USD
}

@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    
    
    """
    Converts an amount from one currency to another using hardcoded exchange rates.

    Args:
        amount: The amount of money to convert.
        from_currency: Source currency code, such as USD, PKR, INR, EUR, or GBP.
        to_currency: Target currency code, such as USD, PKR, INR, EUR, or GBP.

    Returns:
        A formatted string showing the converted amount.
    """
    
    
    
    from_currency = from_currency.strip().upper()
    to_currency = to_currency.strip().upper()
    
    
    amount = str(amount).replace("$", "").replace(",", "").strip()
    amount = float(amount)
    
    if amount <= 0:
        return f"Error: Amount must be greater than 0"
    
    
    if from_currency not in EXCHANGE_RATES_TO_USD:
        return f"Error: We have not any information about this currency {from_currency}"
    
    
    if to_currency not in EXCHANGE_RATES_TO_USD:
        return f"Error: We have not any information about this currency {to_currency}"
    
    
    result_amount = amount * EXCHANGE_RATES_TO_USD[to_currency]
    converted_amount = result_amount / EXCHANGE_RATES_TO_USD[from_currency]
    
    
     
    return ( 
        f"{amount} {from_currency} = "
        f"{converted_amount:.0f} {to_currency}"
    
    )
    
    

BUDGETS = {
    "FOOD": 20000,
    "TRANSPORT": 8000,
    "ENTERTAINMENT": 5000,
    "SHOPPING": 15000,
}

SPENT = {
    "FOOD": 7500,
    "TRANSPORT": 2000,
    "ENTERTAINMENT": 0,
    "SHOPPING": 0,
}


@tool
def get_budget_status(category: str) -> str:
    """
    Returns the remaining budget for a spending category.

    Args:
        category: Budget category. Must be food, transport, entertainment, or shopping.

    Returns:
        Remaining budget, total budget, and amount spent for the category.
    """
    category = category.strip().upper()

    if category not in BUDGETS:
        return (
            "Error: Invalid category. "
            "Use food, transport, entertainment, or shopping."
        )

    total_budget = BUDGETS[category]
    spent = SPENT.get(category, 0)
    remaining = total_budget - spent

    if remaining < 0:
        status = "You have exceeded your budget."
    elif remaining == 0:
        status = "You have used your entire budget."
    else:
        status = "You still have budget remaining."

    return (
        f"Category: {category.title()}\n"
        f"Total budget: {total_budget:.2f} PKR\n"
        f"Spent: {spent:.2f} PKR\n"
        f"Remaining: {remaining:.2f} PKR\n"
        f"Status: {status}"
    )    
    
    
    


@tool
def calculate_savings_goal(target_amount: float, monthly_savings: float) -> str:
    """
    Calculates how many months are needed to reach a savings goal.

    Args:
        target_amount: Total amount you want to save.
        monthly_savings: Amount you can save each month.

    Returns:
        Number of full months required to reach the target savings goal.
    """
    # Clean numeric strings such as "50,000" or "$1,000.50"
    target_amount = float(str(target_amount).replace("$", "").replace(",", "").strip())
    monthly_savings = float(str(monthly_savings).replace("$", "").replace(",", "").strip())

    if target_amount <= 0:
        return "Error: Target amount must be greater than 0."

    if monthly_savings <= 0:
        return "Error: Monthly savings must be greater than 0."

    months_needed = math.ceil(target_amount / monthly_savings)

    return (
        f"To save {target_amount:.2f} PKR "
        f"by saving {monthly_savings:.2f} PKR per month, "
        f"you need {months_needed} month(s)."
    )    
    

SPENDING_TIPS = {
    "FOOD": (
        "Cook more meals at home, plan weekly groceries, and avoid frequent "
        "food delivery. Set a weekly food limit and track every meal expense."
    ),
    "TRANSPORT": (
        "Use public transport, combine errands into one trip, and compare "
        "ride-hailing prices. Walk or cycle for short distances when possible."
    ),
    "ENTERTAINMENT": (
        "Cancel unused subscriptions, use free community events, and set a "
        "monthly entertainment limit before going out."
    ),
    "SHOPPING": (
        "Use the 24-hour rule before buying non-essential items, make a shopping "
        "list, and avoid impulse purchases during sales."
    ),
}

GENERAL_TIP = (
    "Track your expenses daily, set a monthly spending limit, and review your "
    "largest expenses at the end of each week."
)


@tool
def get_spending_tip(category: str) -> str:
    """
    Return a money-saving tip for a spending category.

    Args:
        category: The category where the user overspends, such as food,
            transport, entertainment, or shopping.

    Returns:
        A practical money-saving tip for the given category.
    """
    category = category.strip().upper()

    tip = SPENDING_TIPS.get(category, GENERAL_TIP)

    return f"Money-saving tip for {category.title()}: {tip}"    

print('✅ Tools created successfully!')



In [ ]:
def calling_tools(tool_func, **kwargs):

    try:
        return tool_func.invoke(kwargs)

    except Exception as e:
        return f'Error: str{e}'
    
    
    

print('='*60)
print('TESTING TOOLS DIRECTLY')
print('='*60)

print('\n📊 Formatted Expense Entry:')

print(f"Calculate_Expense: {calling_tools(calculate_expense, amount='5000.0',category='Food',description='Lunch at Restaurant' )}")

print('\n📊 Converted amount:')

print(f"{calling_tools(convert_currency, amount='456.87', from_currency='USD' , to_currency='INR')}")

print('\n📊 Current Budget Status:\n')
print(f"{calling_tools(get_budget_status, category='FOOD')}")

print('\n📊 Savings Goals:\n')

print(f"{calling_tools(calculate_savings_goal, target_amount='18000', monthly_savings='1500')}")

print('\n📊 Get Spending Tip:\n')

print(f"{calling_tools(get_spending_tip, category='ENTERTAINMENT')}")

In [ ]:
print('🤖 Initializing AWS Bedrock LLM...\n')

llm = ChatBedrockConverse(
model_id = MODEL_ID,
region_name = AWS_REGION,
temperature = 0.7,
max_tokens = 512    

)

result = llm.invoke([HumanMessage(content= 'AWS Bedrock is ready to run !')])
print('✅ LLM initialized successfully!')
print(f'   Model  : {MODEL_ID}')
print(f'   Region : {AWS_REGION}')
print(f'   Result   : {result.content[:30]}...')





In [ ]:

# create the finance agent

tools = [
    calculate_expense,
    convert_currency,
    get_budget_status,
    calculate_savings_goal,
    get_spending_tip  
    
]


my_finance_agent = create_agent(

model = llm,
tools = tools,
system_prompt = "You are AI Powered Personal Finance Assistant. You are here to solving our financial problems."    


)

print('\n✅ Agent created! (LangChain 1.x)')



In [ ]:

def run_finance_agent(query: str) -> str:
    
    answer= my_finance_agent.invoke({ 'messages': [{'role': 'user' , 'content': query }]})
    return answer['messages'][-1].content   
    


print('✅ Finance Agent is ready to run !')
print('   Call: run_finance_agent("give here the question!")')


In [ ]:



print('\n' + '='*70)
print('AGENT EXECUTION - Formatted THE Expense Entry QUERY')
print('='*70)

query = 'Add an expense: amount is 5000.0, category is Food, and description is Lunch at Restaurant.'
print(f'\n🔍 User Query: {query}\n')

try:
    answer = run_finance_agent(query)
    print(f'\n✅ Final Answer: {answer}')
except Exception as e:
    print(f'⚠️ Error: {e}')






In [ ]:


print('\n' + '='*70)
print('AGENT EXECUTION - Convert Currency QUERY')
print('='*70)

query = 'Convert my currency: amount is 125000, from_currency is USD , to_currency is INR.'
print(f'\n🔍 User Query: {query}\n')

try:
    answer = run_finance_agent(query)
    print(f'\n✅ Final Answer: {answer}')
except Exception as e:
    print(f'⚠️ Error: {e}')


In [ ]:


print('\n' + '='*70)
print('AGENT EXECUTION - Budget Status QUERY')
print('='*70)

query = 'Give me my Budget Status: category is TRANSPORT.'
print(f'\n🔍 User Query: {query}\n')

try:
    answer = run_finance_agent(query)
    print(f'\n✅ Final Answer: {answer}')
except Exception as e:
    print(f'⚠️ Error: {e}')


In [ ]:


print('\n' + '='*70)
print('AGENT EXECUTION - Formatted THE Expense Entry QUERY')
print('='*70)

query = 'Calculate my Saving Goals: target_amount is 79877 , monthly_savings is 500.'
print(f'\n🔍 User Query: {query}\n')

try:
    answer = run_finance_agent(query)
    print(f'\n✅ Final Answer: {answer}')
except Exception as e:
    print(f'⚠️ Error: {e}')


In [ ]:


print('\n' + '='*70)
print('AGENT EXECUTION - Spending Tip QUERY')
print('='*70)

query = 'Give me the spending tip: my category is SHOPPING .'
print(f'\n🔍 User Query: {query}\n')

try:
    answer = run_finance_agent(query)
    print(f'\n✅ Final Answer: {answer}')
except Exception as e:
    print(f'⚠️ Error: {e}')
